In [1]:
from cobra.io import read_sbml_model
import requests
import os
import pandas as pd

In [2]:
ecoli_model = read_sbml_model('/Users/mayaabdalla/Downloads/iML1515.xml')

In [11]:
ecoli_med = ecoli_model.medium

In [12]:
print(ecoli_med)

{'EX_pi_e': 1000.0, 'EX_co2_e': 1000.0, 'EX_fe3_e': 1000.0, 'EX_h_e': 1000.0, 'EX_mn2_e': 1000.0, 'EX_fe2_e': 1000.0, 'EX_glc__D_e': 10.0, 'EX_zn2_e': 1000.0, 'EX_mg2_e': 1000.0, 'EX_ca2_e': 1000.0, 'EX_ni2_e': 1000.0, 'EX_cu2_e': 1000.0, 'EX_sel_e': 1000.0, 'EX_cobalt2_e': 1000.0, 'EX_h2o_e': 1000.0, 'EX_mobd_e': 1000.0, 'EX_so4_e': 1000.0, 'EX_nh4_e': 1000.0, 'EX_k_e': 1000.0, 'EX_na1_e': 1000.0, 'EX_cl_e': 1000.0, 'EX_o2_e': 1000.0, 'EX_tungs_e': 1000.0, 'EX_slnt_e': 1000.0}



Minimal uptake rates the model needs to achieve exactly 0.1 h^-1 growth. The default optimum is ~0.877 h^-1. At that rate, we would need the glucose demand to be ~10X higher, why the default sets glucose to 10 mmol/gDW/h. Minimal media values lower bound, not an operating point.

In [13]:
# ── Default medium (M9-like minimal) ─────────────────────────────────────────
print(f"Default medium: {len(ecoli_model.medium)} components\n")
print(f"{'Exchange reaction':<25} {'Uptake bound':>14}  Metabolite name")
print("-" * 75)
for rxn_id, bound in sorted(ecoli_model.medium.items(), key=lambda x: x[1], reverse=True):
    rxn = ecoli_model.reactions.get_by_id(rxn_id)
    ext_mets = [m for m in rxn.metabolites if m.compartment == 'e']
    met_name = ext_mets[0].name if ext_mets else rxn.name
    print(f"  {rxn_id:<23} {bound:>14.1f}  {met_name}")

# ── cobra computed minimal medium ────────────────────────────────────────────
print("\n" + "="*75)
print("Cobra-computed minimal medium (minimum uptake flux to sustain growth):")
print("="*75)
from cobra.medium import minimal_medium
min_med = minimal_medium(ecoli_model, min_objective_value=0.1)
print(f"\n{len(min_med)} components required:\n")
print(f"{'Exchange reaction':<25} {'Min flux':>10}  Metabolite name")
print("-" * 60)
for rxn_id, flux in min_med.sort_values(ascending=False).items():
    rxn = ecoli_model.reactions.get_by_id(rxn_id)
    ext_mets = [m for m in rxn.metabolites if m.compartment == 'e']
    met_name = ext_mets[0].name if ext_mets else rxn.name
    print(f"  {rxn_id:<23} {flux:>10.4f}  {met_name}")

Default medium: 24 components

Exchange reaction           Uptake bound  Metabolite name
---------------------------------------------------------------------------
  EX_pi_e                         1000.0  Phosphate
  EX_co2_e                        1000.0  CO2 CO2
  EX_fe3_e                        1000.0  Iron (Fe3+)
  EX_h_e                          1000.0  H+
  EX_mn2_e                        1000.0  Manganese
  EX_fe2_e                        1000.0  Fe2+ mitochondria
  EX_zn2_e                        1000.0  Zinc
  EX_mg2_e                        1000.0  Magnesium
  EX_ca2_e                        1000.0  Calcium
  EX_ni2_e                        1000.0  Nickel
  EX_cu2_e                        1000.0  Copper
  EX_sel_e                        1000.0  Selenate
  EX_cobalt2_e                    1000.0  Co2+
  EX_h2o_e                        1000.0  H2O H2O
  EX_mobd_e                       1000.0  Molybdate
  EX_so4_e                        1000.0  Sulfate
  EX_nh4_e               

In [ ]:
# ── Convert collaborator LB recipe to mmol/L ─────────────────────────────────
# Recipe: 10g/L tryptone + 5g/L yeast extract + 10g/L NaCl

# Tryptone amino acid composition (g per 100g) -- casein pancreatic digest
# Source: standard casein hydrolysate analysis
tryptone_aa = {
    'ala__L': (3.0,  89.09),
    'arg__L': (3.4, 174.20),
    'asp__L': (4.8, 133.10),   # ~70% of total Asp+Asn
    'asn__L': (2.1, 132.12),   # ~30% of total Asp+Asn
    'cys__L': (0.4, 121.16),
    'glu__L': (17.2,147.13),   # ~80% of total Glu+Gln
    'gln__L': (4.3, 146.15),   # ~20% of total Glu+Gln
    'gly':    (1.8,  75.03),
    'his__L': (2.9, 155.16),
    'ile__L': (5.4, 131.17),
    'leu__L': (9.3, 131.17),
    'lys__L': (7.8, 146.19),
    'met__L': (2.8, 149.21),
    'phe__L': (5.1, 165.19),
    'pro__L': (9.6, 115.13),
    'ser__L': (5.4, 105.09),
    'thr__L': (4.3, 119.12),
    'trp__L': (1.2, 204.23),
    'tyr__L': (5.4, 181.19),
    'val__L': (6.8, 117.15),
}

# Yeast extract amino acid composition (g per 100g yeast protein, ~40% of dry weight)
# Based on S. cerevisiae protein profile
yeast_aa = {
    'ala__L': (7.5,  89.09),
    'arg__L': (5.7, 174.20),
    'asp__L': (7.3, 133.10),
    'asn__L': (3.1, 132.12),
    'cys__L': (1.2, 121.16),
    'glu__L': (9.2, 147.13),
    'gln__L': (4.0, 146.15),
    'gly':    (4.5,  75.03),
    'his__L': (2.3, 155.16),
    'ile__L': (5.0, 131.17),
    'leu__L': (8.0, 131.17),
    'lys__L': (7.8, 146.19),
    'met__L': (1.8, 149.21),
    'phe__L': (4.7, 165.19),
    'pro__L': (4.0, 115.13),
    'ser__L': (5.2, 105.09),
    'thr__L': (5.5, 119.12),
    'trp__L': (1.2, 204.23),
    'tyr__L': (3.8, 181.19),
    'val__L': (6.2, 117.15),
}
yeast_aa_fraction = 0.40   # 40% of yeast extract is amino acids

# Calculate mmol/L for each amino acid
tryptone_g = 10.0
yeast_g    = 5.0
nacl_g     = 10.0

aa_mmol_L = {}
for aa, (pct, mw) in tryptone_aa.items():
    aa_mmol_L[aa] = aa_mmol_L.get(aa, 0) + (tryptone_g * pct/100) / mw * 1000

for aa, (pct, mw) in yeast_aa.items():
    aa_mmol_L[aa] = aa_mmol_L.get(aa, 0) + (yeast_g * yeast_aa_fraction * pct/100) / mw * 1000

nacl_mmol_L = (nacl_g / 58.44) * 1000   # MW NaCl = 58.44

print("Recipe: 10g tryptone + 5g yeast extract + 10g NaCl per liter")
print()
print(f"{'Amino acid':<12} {'mmol/L':>8}   (from tryptone + yeast extract)")
print("-" * 50)
for aa, mmol in sorted(aa_mmol_L.items(), key=lambda x: -x[1]):
    ex_id = f"EX_{aa}_e" if aa != 'gly' else "EX_gly_e"
    print(f"  {aa:<12} {mmol:>8.2f}   → {ex_id}")
print()
print(f"NaCl:  {nacl_mmol_L:.1f} mM → Na⁺ {nacl_mmol_L:.1f} mM, Cl⁻ {nacl_mmol_L:.1f} mM")

# ── Build LB medium dict using calculated mmol/L as exchange bounds ───────────
# Exchange bound (mmol/gDW/h) = concentration (mmol/L) at early growth
# when cell density << 1 gDW/L, so all nutrients are effectively non-limiting
lb_medium_collab = {
    # Calculated from recipe
    **{f"EX_{aa}_e": round(mmol, 2) for aa, mmol in aa_mmol_L.items()},
    "EX_na1_e":     round(nacl_mmol_L, 1),
    "EX_cl_e":      round(nacl_mmol_L, 1),
    # Inorganic / environmental (open -- not limiting in LB)
    "EX_h2o_e":     1000.0,
    "EX_h_e":       1000.0,
    "EX_co2_e":     1000.0,
    "EX_pi_e":      1000.0,
    "EX_o2_e":        20.0,   # aerobic
    "EX_nh4_e":     1000.0,
    "EX_so4_e":     1000.0,
    "EX_k_e":       1000.0,
    "EX_mg2_e":     1000.0,
    "EX_fe2_e":     1000.0,
    "EX_fe3_e":     1000.0,
    "EX_ca2_e":     1000.0,
    "EX_mn2_e":     1000.0,
    "EX_zn2_e":     1000.0,
    "EX_cu2_e":     1000.0,
    "EX_ni2_e":     1000.0,
    "EX_cobalt2_e": 1000.0,
    "EX_mobd_e":    1000.0,
    # Vitamins present in iML1515 (riboflavin/folate not in model exchanges)
    "EX_thm_e":     1.0,    # thiamine
    "EX_nac_e":     1.0,    # niacin
    "EX_pnto__R_e": 1.0,    # pantothenate
    "EX_pydx_e":    1.0,    # pyridoxal
    "EX_btn_e":     1.0,    # biotin
}
# fix gly key (no stereoisomer suffix)
if 'EX_gly_e' not in lb_medium_collab and 'EX_gly__L_e' in lb_medium_collab:
    lb_medium_collab['EX_gly_e'] = lb_medium_collab.pop('EX_gly__L_e')

# ── Run FBA and compare ───────────────────────────────────────────────────────
with ecoli_model:
    sol_m9 = ecoli_model.optimize()
    print(f"\nM9 minimal (glucose):   growth = {sol_m9.objective_value:.4f} h⁻¹")

with ecoli_model:
    ecoli_model.medium = lb_medium_collab
    sol_lb = ecoli_model.optimize()
    print(f"LB (collab recipe):     growth = {sol_lb.objective_value:.4f} h⁻¹")

print("\nAmino acids actually consumed in LB (non-zero uptake):")
print(f"  {'Exchange':<22} {'Bound (mmol/L)':>15}  {'Actual flux':>12}")
print("-" * 55)
for aa in sorted(aa_mmol_L):
    rxn_id = f"EX_{aa}_e"
    bound = lb_medium_collab.get(rxn_id, 0)
    flux  = sol_lb.fluxes.get(rxn_id, 0)
    if abs(flux) > 1e-6:
        print(f"  {rxn_id:<22} {bound:>15.2f}  {flux:>12.4f}")

print(f"\n  EX_na1_e (Na⁺ from NaCl)  bound={nacl_mmol_L:.1f}  flux={sol_lb.fluxes.get('EX_na1_e',0):.4f}")